# 06 · Generate the evaluation benchmark

> **Run order.** This notebook is step 6 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


The payoff for choosing this domain.

Reported financials give values we know to be true. Locating each one inside its
own annual report yields an auto-labelled **(question, answer, source page)**
triple. No hand-written questions, and **no LLM call anywhere in this notebook**.

**The false-positive fix.** Notebook 05 measured a raw match rate of 47–57% and
flagged it as a ceiling, because a 4-digit figure can collide by chance somewhere
in 300 pages. The fix is proximity: a number is accepted only when a *label* for
its concept appears in the **same extracted element**. Elements are already
page-level blocks and whole tables, so that is a natural and strict window — a
revenue figure must sit in the revenue row, not merely somewhere on the page.

That trades coverage for trustworthiness, which is the correct direction.

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from analyst.benchmark import CONCEPT_ALIASES

print(f"{len(CONCEPT_ALIASES)} concepts are anchorable.")
print("Yahoo's taxonomy on the left, what an Indian annual report actually prints on the right:\n")
pd.DataFrame([
    {"yahoo_concept": k, "report_labels": ", ".join(v)}
    for k, v in list(CONCEPT_ALIASES.items())[:8]
])

## Two matching passes

**exact** — the value appears verbatim in some printed form.

**tolerant** — a number within 0.5% appears. This exists because the vendor and the filing rarely agree to the last rupee: HDFC Bank's FY2025 net profit is ₹67,347.36 crore in the filing and ₹67,351 crore at the vendor. Exact matching found *nothing* for HDFCBANK; tolerance recovers it.

In [ ]:
from decimal import Decimal
from analyst.numfmt import candidate_strings, find_value_tolerant

value = Decimal("520412500000")   # SUNPHARMA FY2025 total revenue, absolute INR
forms = sorted(candidate_strings(value), key=len, reverse=True)
print(f"{len(forms)} plausible printed forms of the same number, e.g.:")
print("  ", forms[:8])

print("\nTolerant match against a filing that rounds differently:")
print("  ", find_value_tolerant(Decimal("673510000000"), "Profit for the year 67,347.36"))
print("  (token, relative error) - well inside the 0.5% band")

## Anchor every fact we can

In [ ]:
from sqlalchemy import select
from analyst.benchmark import (
    BenchmarkQuestion, ElementView, build_growth_questions, build_questions, fy_window,
)
from analyst.config import get_settings
from analyst.db import session_scope
from analyst.models import Company, Document, ElementRow, Fact

MIN_ABS_VALUE = Decimal(10**5)
settings = get_settings()

all_questions: list[BenchmarkQuestion] = []
per_ticker: dict[str, list[BenchmarkQuestion]] = {}
report = []

with session_scope() as s:
    names = {c.ticker: c.name for c in s.execute(select(Company)).scalars().all()}
    doc_rows = [(d.document_id, d.ticker, d.fiscal_year)
                for d in s.execute(select(Document).order_by(Document.ticker,
                                                             Document.fiscal_year)).scalars().all()]
    for document_id, ticker, fy in doc_rows:
        facts = [(c, v, u) for c, v, u in s.execute(
            select(Fact.concept, Fact.value, Fact.unit)
            .where(Fact.ticker == ticker)
            .where(Fact.concept.in_(CONCEPT_ALIASES.keys()))
            .where(Fact.period_end.between(*fy_window(fy)))
        ).all() if abs(v) >= MIN_ABS_VALUE]

        elements = [ElementView(element_id=e[0], page=e[1], type=e[2], text=e[3] or "")
                    for e in s.execute(
                        select(ElementRow.element_id, ElementRow.page, ElementRow.type,
                               ElementRow.text)
                        .where(ElementRow.document_id == document_id)
                        .where(ElementRow.text.is_not(None))
                        .where(ElementRow.type.in_(("table", "text")))
                        .order_by(ElementRow.page, ElementRow.seq)).all()]

        qs = build_questions(ticker, names.get(ticker, ticker), fy, document_id, facts, elements)
        all_questions.extend(qs)
        per_ticker.setdefault(ticker, []).extend(qs)
        report.append({"ticker": ticker, "fy": fy, "candidates": len(facts),
                       "anchored": len(qs), "elements": len(elements)})

pd.DataFrame(report)

## Multi-hop questions come free

The same concept anchored in two fiscal years becomes a growth question. Both years are already anchored, so a growth question can never be less trustworthy than the lookups it is built from.

In [ ]:
import json

growth = []
for ticker, qs in per_ticker.items():
    growth.extend(build_growth_questions(ticker, names.get(ticker, ticker), qs))
all_questions.extend(growth)

out_path = settings.data_dir / "benchmark" / "questions.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("w", encoding="utf-8") as fh:
    for q in all_questions:
        fh.write(json.dumps(q.model_dump(), ensure_ascii=False) + "\n")

df = pd.DataFrame([q.model_dump() for q in all_questions])
print(f"BENCHMARK SIZE: {len(df)} questions -> {out_path}\n")
print(df.groupby(["question_type", "match_kind"]).size().to_string())
df[["question", "answer_display", "expected_pages", "match_kind", "evidence_type"]].head(10)

## What an anchored question looks like

In [ ]:
for q in all_questions[:5]:
    print(f"Q  {q.question}")
    print(f"A  {q.answer_display}    (matched as '{q.matched_as}', {q.match_kind})")
    print(f"-> {q.expected_element_ids[0]}  page {q.expected_pages[0]}\n")